In [ ]:
# imports
import numpy as np
import pandas as pd
import sys
from scipy import stats

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.ticker import StrMethodFormatter
plt.style.use('scifigs.mplstyle')
SAVE_FIG = False

sys.path.append('helpers/pcca_fa/')
from dual_pfc_funcs import load_dict, getParams, get_top_angle
import helpers.pcca_fa.pcca_fa_mdl as pf
color_map = getParams()['color_map']

In [ ]:
fname = 'preprocessed_data/simdataset_varyThetaSubsample.pkl'
dat = load_dict(fname)
n_boots = dat['n_boots']
thetas = dat['thetas']

df_subsample = pd.DataFrame(columns=['Theta','Base-Theta_X1','GT-Theta_X1','Est-Theta_X1','Error-Theta_X1'])
# Theta - ground truth value in full space (theta_full), same for many simulations
# Base-Theta - ground truth value in subsampled space (theta_sub), same for many simulations
# GT-Theta - ground truth value in neural space (theta_sim), unique per simulation
# Est-Theta - estimated value in neural space (theta_estimated), unique per simulation

counter = 0
for (theta,base_param) in zip(thetas,dat['base_params']):
    base_angle,_ = get_top_angle(base_param)
    for boot in range(n_boots):
        gt_param = dat['sim_params'][counter]
        gt_angle,_ = get_top_angle(gt_param)

        est_param = dat['est_params'][counter]
        est_angle,_ = get_top_angle(est_param)

        df2 = {'Theta':theta,'Base-Theta_X1':base_angle,'GT-Theta_X1':gt_angle,'Est-Theta_X1':est_angle,'Error-Theta_X1':est_angle-gt_angle}
        df_subsample.loc[len(df_subsample)] = df2
        counter += 1
# df_subsample

In [ ]:
# determine whether subselecting/swapping out single neurons from the population yields a wide angle distribution
thetas_to_plot = [60,90]
bins = np.arange(0,95,4)
colors = ['.6','.3'] # to match Fig 5d
fig,ax = plt.subplots(2,1,constrained_layout=True,sharey=True)
for i,theta in enumerate(thetas_to_plot):
    tmp_df = df_subsample[df_subsample['Theta']==theta]
    base_angle = tmp_df.iloc[0]['Base-Theta_X1']

    sim_angles = tmp_df['GT-Theta_X1'].to_numpy()
    est_angles = tmp_df['Est-Theta_X1'].to_numpy()
    
    # plot
    ax[0].hist(sim_angles,bins=bins,density=True,color=colors[i],alpha=0.5)
    # ax[0].axvline(base_angle,linestyle='--',color=colors[i])
    ax[0].plot(sim_angles.mean(),0.07,marker='v',color=colors[i],ms=6)

    ax[1].hist(est_angles,bins=bins,density=True,color=colors[i],alpha=0.5)
    # ax[1].axvline(base_angle,linestyle='--',color=colors[i])
    ax[1].plot(est_angles.mean(),0.065,marker='v',color=colors[i],ms=6)
    
    print('theta_full = {}deg, theta_sub = {:.2f}deg'.format(theta,base_angle))
    print('mean of theta_sim = {:.2f}deg, mean of theta_est = {:.2f}deg'.format(sim_angles.mean(),est_angles.mean()))
ax[0].set_xlim([0,91])
ax[0].xaxis.set_major_formatter(StrMethodFormatter(u"{x:.0f}°"))
ax[0].set_xlabel(r'$\theta_{sim}$')
ax[0].set_yticks([])
ax[0].set_xticks(np.arange(0,91,30))
ax[1].set_xlim([0,91])
ax[1].set_xticks(np.arange(0,91,30))
ax[1].set_yticks([])
ax[1].set_xlabel(r'$\theta_{estimated}$')
ax[1].xaxis.set_major_formatter(StrMethodFormatter(u"{x:.0f}°"))

fig.supylabel('# of sim. populations', fontsize=12)
fig.suptitle('Subsampled simulated data')

if SAVE_FIG:
    pdf = PdfPages('figs/angles_sim_subsampled.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

In [ ]:
# stats
# load data from fig 5 for comparison
dat = load_dict('preprocessed_data/simdataset_varyThetaShuffle.pkl')
cols = ['SessionName'] + list(list(dat.values())[0].keys())
df = pd.DataFrame(columns=cols)
for i, (sess, curr_dat) in enumerate(dat.items()):    
    df2 = {'SessionName':sess,**curr_dat}
    df.loc[len(df)] = df2
# df

for i,theta in enumerate(thetas_to_plot):
    tmp_df = df_subsample[df_subsample['Theta']==theta]

    sim_angles = tmp_df['GT-Theta_X1'].to_numpy()
    est_angles = tmp_df['Est-Theta_X1'].to_numpy()

    p = stats.levene(sim_angles, est_angles).pvalue
    print('Levene test against subsampled dists for theta = {}deg: p = {:.6f}'.format(theta,p))

    orig_est_angles = df['fit_theta{}_x1'.format(theta)]
    p = stats.levene(orig_est_angles, est_angles).pvalue
    print('Levene test against original dists for theta = {}deg: p = {:.6f}'.format(theta,p))